In [2]:
import pandas as pd
import numpy as np

# โหลดข้อมูล (แก้ path ให้ตรงกับเครื่องของคุณ)
train_t1 = pd.read_csv('train.csv')
test_t1 = pd.read_csv('test.csv')

print("=== 1. ขนาดข้อมูล ===")
print(f"Train: {train_t1.shape} | Test: {test_t1.shape}")

print("\n=== 2. สัดส่วนเฉลย is_cheater (รวมค่าว่าง) ===")
print(train_t1['is_cheater'].value_counts(dropna=False))

print("\n=== 3. เปรียบเทียบค่าเฉลี่ยระหว่าง คนปกติ (0) vs คนโกง (1) ===")
summary = train_t1.groupby('is_cheater').mean(numeric_only=True).T
summary['diff_ratio'] = summary[1.0] / (summary[0.0] + 1e-5)
print(summary.sort_values(by='diff_ratio', ascending=False).round(3))

=== 1. ขนาดข้อมูล ===
Train: (104000, 33) | Test: (25889, 32)

=== 2. สัดส่วนเฉลย is_cheater (รวมค่าว่าง) ===
is_cheater
0.0    59091
1.0    31725
NaN    13184
Name: count, dtype: int64

=== 3. เปรียบเทียบค่าเฉลี่ยระหว่าง คนปกติ (0) vs คนโกง (1) ===
is_cheater                   0.0      1.0  diff_ratio
reaction_time_ms         133.953  137.670       1.028
device_changes_count       7.438    7.474       1.005
night_play_ratio           0.378    0.380       1.005
team_play_score            0.644    0.646       1.003
clutch_success_rate        0.340    0.340       1.002
movement_pattern_score     0.714    0.715       1.001
weapon_switch_speed        0.634    0.634       1.000
communication_rate         0.550    0.550       1.000
sessions_per_day           4.945    4.947       1.000
utility_usage_rate         0.581    0.581       1.000
survival_time_avg         66.239   66.183       0.999
map_knowledge              0.690    0.689       0.999
buy_decision_score         0.598    0.597       

In [3]:
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import fbeta_score

# 1. เตรียมข้อมูลเฉพาะแถวที่มีเฉลย
train_labeled_t1 = train_t1.dropna(subset=['is_cheater']).reset_index(drop=True)
features_t1 = [col for col in test_t1.columns if col != 'player_id']

X_t1 = train_labeled_t1[features_t1]
y_t1 = train_labeled_t1['is_cheater'].astype(int)

# 2. เทรน Baseline 5-Fold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds_t1 = np.zeros(len(train_labeled_t1))
importances = np.zeros(len(features_t1))

lgb_params = {
    'objective': 'binary',
    'metric': 'binary_logloss',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'n_estimators': 1000,
    'random_state': 42,
    'verbose': -1
}

for fold, (train_idx, val_idx) in enumerate(skf.split(X_t1, y_t1)):
    X_tr, y_tr = X_t1.iloc[train_idx], y_t1.iloc[train_idx]
    X_va, y_va = X_t1.iloc[val_idx], y_t1.iloc[val_idx]

    model_t1 = lgb.LGBMClassifier(**lgb_params)
    model_t1.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )

    oof_preds_t1[val_idx] = model_t1.predict_proba(X_va)[:, 1]
    importances += model_t1.feature_importances_ / skf.n_splits

# 3. หา Threshold ที่ดีที่สุดสำหรับ Macro F2
best_thresh_t1 = 0.5
best_f2_t1 = 0.0
for thresh in np.arange(0.10, 0.90, 0.01):
    score = fbeta_score(y_t1, (oof_preds_t1 >= thresh).astype(int), beta=2, average='macro')
    if score > best_f2_t1:
        best_f2_t1 = score
        best_thresh_t1 = thresh

print(f"=== คะแนน Baseline ตั้งต้น ===")
print(f"Best Threshold : {best_thresh_t1:.2f}")
print(f"Macro F2 Score : {best_f2_t1:.5f}")

print("\n=== Top 15 คอลัมน์ที่โมเดลใช้ตัดสินใจมากที่สุด ===")
imp_df = pd.DataFrame({'feature': features_t1, 'importance': importances})
print(imp_df.sort_values(by='importance', ascending=False).head(15).to_string(index=False))

=== คะแนน Baseline ตั้งต้น ===
Best Threshold : 0.37
Macro F2 Score : 0.77872

=== Top 15 คอลัมน์ที่โมเดลใช้ตัดสินใจมากที่สุด ===
                feature  importance
       account_age_days      1039.0
    friend_network_size      1022.4
       reports_received       653.0
    crosshair_placement       539.6
                  level       491.2
       game_sense_score       449.8
    headshot_percentage       389.0
       kill_death_ratio       379.8
level_progression_speed       376.4
               win_rate       317.4
         accuracy_score       286.0
       kill_consistency       285.0
    spray_control_score       274.4
     buy_decision_score       265.4
      aiming_smoothness       262.4


In [4]:
def add_features(df):
    df = df.copy()

    # 1. สัญญาณความน่าเชื่อถือของบัญชี (Account Credibility)
    df['friends_per_day'] = df['friend_network_size'] / (df['account_age_days'] + 1.0)
    df['reports_per_day'] = df['reports_received'] / (df['account_age_days'] + 1.0)
    df['reports_per_level'] = df['reports_received'] / (df['level'] + 1.0)
    df['level_per_day'] = df['level'] / (df['account_age_days'] + 1.0)

    # 2. ความไม่สมดุลระหว่าง "ความแม่น" กับ "สมองเกม" (Aim vs Game Sense)
    df['aim_to_gamesense_ratio'] = (df['headshot_percentage'] + df['accuracy_score']) / (df['game_sense_score'] + df['map_knowledge'] + 1e-5)
    df['kd_to_winrate_ratio'] = df['kill_death_ratio'] / (df['win_rate'] + 1.0)

    # 3. จำนวนค่าว่าง (Missing Values) ต่อผู้เล่น 1 คน
    base_cols = [c for c in test_t1.columns if c != 'player_id']
    df['missing_count'] = df[base_cols].isnull().sum(axis=1)

    return df

# สร้างชุดข้อมูลใหม่ที่มี Feature Engineering
train_fe_t1 = add_features(train_labeled_t1)
features_fe_t1 = [col for col in train_fe_t1.columns if col not in ['player_id', 'is_cheater']]

X_fe_t1 = train_fe_t1[features_fe_t1]
y_fe_t1 = train_fe_t1['is_cheater'].astype(int)

# เทรน 5-Fold ด้วยพารามิเตอร์เดิมทุกอย่าง เพื่อเทียบคะแนนแบบยุติธรรม (Apples-to-Apples)
oof_fe_t1 = np.zeros(len(train_fe_t1))
imp_fe = np.zeros(len(features_fe_t1))

for fold, (train_idx, val_idx) in enumerate(skf.split(X_fe_t1, y_fe_t1)):
    X_tr, y_tr = X_fe_t1.iloc[train_idx], y_fe_t1.iloc[train_idx]
    X_va, y_va = X_fe_t1.iloc[val_idx], y_fe_t1.iloc[val_idx]

    model_fe = lgb.LGBMClassifier(**lgb_params)
    model_fe.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )

    oof_fe_t1[val_idx] = model_fe.predict_proba(X_va)[:, 1]
    imp_fe += model_fe.feature_importances_ / skf.n_splits

best_thresh_fe = 0.5
best_f2_fe = 0.0
for thresh in np.arange(0.10, 0.90, 0.01):
    score = fbeta_score(y_fe_t1, (oof_fe_t1 >= thresh).astype(int), beta=2, average='macro')
    if score > best_f2_fe:
        best_f2_fe = score
        best_thresh_fe = thresh

print(f"=== เปรียบเทียบคะแนนหลังทำ Feature Engineering ===")
print(f"คะแนน Baseline เดิม : {best_f2_t1:.5f}")
print(f"คะแนนรอบใหม่ (FE)  : {best_f2_fe:.5f} (เปลี่ยนแปลง: {best_f2_fe - best_f2_t1:+.5f})")
print(f"Best Threshold ใหม่ : {best_thresh_fe:.2f}")

print("\n=== Top 15 คอลัมน์สำคัญรอบใหม่ ===")
imp_fe_df = pd.DataFrame({'feature': features_fe_t1, 'importance': imp_fe})
print(imp_fe_df.sort_values(by='importance', ascending=False).head(15).to_string(index=False))

=== เปรียบเทียบคะแนนหลังทำ Feature Engineering ===
คะแนน Baseline เดิม : 0.77872
คะแนนรอบใหม่ (FE)  : 0.77885 (เปลี่ยนแปลง: +0.00013)
Best Threshold ใหม่ : 0.40

=== Top 15 คอลัมน์สำคัญรอบใหม่ ===
                feature  importance
    friend_network_size       968.0
       account_age_days       901.6
       reports_received       559.2
    crosshair_placement       551.8
                  level       445.2
       game_sense_score       439.6
    headshot_percentage       356.4
level_progression_speed       341.0
       kill_death_ratio       317.2
               win_rate       306.6
       kill_consistency       282.6
    spray_control_score       282.0
      aiming_smoothness       269.2
         accuracy_score       262.4
       reaction_time_ms       258.8


In [6]:
from sklearn.metrics import fbeta_score

# 1. โหลดไฟล์ submission ของทีม
SUB_PATH = 'submission_final.csv'
sub_before = pd.read_csv(SUB_PATH)

# 2. เตรียมข้อมูล Test ด้วยฟังก์ชัน add_features จาก Cell 3
test_fe_t1 = add_features(test_t1)
X_test_fe_t1 = test_fe_t1[features_fe_t1]

assert len(X_test_fe_t1) == len(sub_before), "จำนวนแถว test ไม่ตรงกับ submission!"

# 3. เทรน 5-Fold และทำนายผลลงชุด Test
oof_final = np.zeros(len(X_fe_t1))
test_preds_final = np.zeros(len(X_test_fe_t1))

for fold, (train_idx, val_idx) in enumerate(skf.split(X_fe_t1, y_fe_t1)):
    X_tr, y_tr = X_fe_t1.iloc[train_idx], y_fe_t1.iloc[train_idx]
    X_va, y_va = X_fe_t1.iloc[val_idx], y_fe_t1.iloc[val_idx]

    model_final = lgb.LGBMClassifier(**lgb_params)
    model_final.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )

    oof_final[val_idx] = model_final.predict_proba(X_va)[:, 1]
    test_preds_final += model_final.predict_proba(X_test_fe_t1)[:, 1] / skf.n_splits

# 4. หา Best Threshold ตามกติกา Macro F2 Score ของ Kaggle
best_thresh_final = 0.5
best_f2_final = 0.0
for thresh in np.arange(0.10, 0.90, 0.01):
    score = fbeta_score(y_fe_t1, (oof_final >= thresh).astype(int), beta=2, average='macro')
    if score > best_f2_final:
        best_f2_final = score
        best_thresh_final = thresh

print(f"Best Threshold (Macro F2): {best_thresh_final:.2f} | Score: {best_f2_final:.5f}")

# 5. บันทึกเฉพาะคอลัมน์ task1 และตรวจสอบความปลอดภัย
sub_after = sub_before.copy()
sub_after['task1'] = (test_preds_final >= best_thresh_final).astype(int)

other_cols = ['id', 'task2', 'task3', 'task4', 'task5']
is_safe = sub_before[other_cols].equals(sub_after[other_cols])

print("คำตอบข้ออื่นของเพื่อนอยู่ครบ 100% ไหม:", is_safe)
print("สัดส่วนคำทำนาย task1:\n", sub_after['task1'].value_counts(normalize=True).round(4))

if is_safe and sub_after['task1'].isnull().sum() == 0:
    sub_after.to_csv(SUB_PATH, index=False)
    print(f"\n[สำเร็จ] บันทึกคำตอบลงใน '{SUB_PATH}' เรียบร้อย พร้อมดาวน์โหลดส่ง Kaggle!")

Best Threshold (Macro F2): 0.40 | Score: 0.77885
คำตอบข้ออื่นของเพื่อนอยู่ครบ 100% ไหม: True
สัดส่วนคำทำนาย task1:
 task1
0    0.6171
1    0.3829
Name: proportion, dtype: float64

[สำเร็จ] บันทึกคำตอบลงใน 'submission_final.csv' เรียบร้อย พร้อมดาวน์โหลดส่ง Kaggle!
